In [1]:
import json
import re
import pandas as pd

# ── STEP 1: Extract nightly price from JSON ──────────────────────────────────

with open("Airbnb_Scrabbed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def extract_nightly_price(listing):
    """Try multiple strategies to get nightly price."""
    # Strategy 1: Parse "N nights x $X.XX" from basePrice description
    try:
        desc = listing["price"]["breakDown"]["basePrice"]["description"]
        match = re.search(r'(\d+)\s*nights?\s*x\s*\$(\d+(?:\.\d+)?)', desc)
        if match:
            return float(match.group(2))
    except (KeyError, TypeError):
        pass

    # Strategy 2: Divide base total by number of nights
    try:
        desc = listing["price"]["breakDown"]["basePrice"]["description"]
        base_total = float(
            listing["price"]["breakDown"]["basePrice"]["price"]
            .replace("$", "").replace(",", "")
        )
        nights_match = re.search(r'(\d+)\s*nights?', desc)
        if nights_match:
            nights = int(nights_match.group(1))
            if nights > 0:
                return round(base_total / nights, 2)
    except (KeyError, TypeError, ZeroDivisionError):
        pass

    # Strategy 3: No nightly price available
    return None

rows = []
for listing in data:
    rows.append({
        "id":            listing.get("id"),
        "url":           listing.get("url"),
        "nightly_price": extract_nightly_price(listing)
    })

json_df = pd.DataFrame(rows)

In [2]:
# ── STEP 2: Clean the extracted data ─────────────────────────────────────────

# Drop rows where nightly price could not be extracted
print(f"Total listings:             {len(json_df)}")
print(f"Missing nightly price:      {json_df['nightly_price'].isna().sum()}")

json_df = json_df.dropna(subset=["nightly_price"])
json_df = json_df[json_df["nightly_price"] > 0]          # remove zero prices
json_df = json_df.drop_duplicates(subset=["id"])          # remove duplicates
json_df["id"] = json_df["id"].astype(str).str.strip()    # normalize id type

print(f"After cleaning:             {len(json_df)}")

Total listings:             833
Missing nightly price:      0
After cleaning:             833


In [3]:
# After STEP 2 cleaning
json_df.to_csv("nightly_prices.csv", index=False)

In [4]:
# ── STEP 3: Merge with the processed CSV ─────────────────────────────────────

csv_df = pd.read_csv("airbnb_processed.csv")
csv_df["id"] = csv_df["id"].astype(str).str.strip()      # normalize id type

merged_df = csv_df.merge(json_df[["id", "nightly_price"]], on="id", how="left")

# Check merge quality
matched   = merged_df["nightly_price"].notna().sum()
unmatched = merged_df["nightly_price"].isna().sum()
print(f"Matched rows:               {matched}")
print(f"Unmatched (no JSON price):  {unmatched}")

merged_df.to_csv("airbnb_merged.csv", index=False)
print("\nSaved → airbnb_merged.csv")
print(merged_df[["id", "url", "price_original", "nightly_price"]].head())

Matched rows:               831
Unmatched (no JSON price):  0

Saved → airbnb_merged.csv
                    id                                                url  \
0  1292713234154945394  https://www.airbnb.com/rooms/12927132341549453...   
1  1508718511630646313  https://www.airbnb.com/rooms/15087185116306463...   
2  1297327219631789358  https://www.airbnb.com/rooms/12973272196317893...   
3  1606001853199411128  https://www.airbnb.com/rooms/16060018531994111...   
4  1314833467096489875  https://www.airbnb.com/rooms/13148334670964898...   

   price_original  nightly_price  
0           750.0         150.00  
1           527.0         105.37  
2           194.0          38.64  
3           593.0         118.49  
4           117.0          23.22  


In [5]:
import pandas as pd

df = pd.read_csv("airbnb_merged.csv")

# Shape
print("Shape:", df.shape)

# Column types
print("\nDtypes:\n", df.dtypes)

# Missing values
print("\nMissing values:\n", df.isnull().sum())

# Duplicates
print("\nDuplicate rows:", df.duplicated().sum())

# Basic stats on numeric columns
print("\nDescribe:\n", df.describe())

# Sample rows
print("\nSample:\n", df.head(3))

Shape: (831, 24)

Dtypes:
 id                      int64
title                  object
url                    object
thumbnail              object
lat                   float64
lng                   float64
rating_overall        float64
reviews_count         float64
rating_accuracy       float64
rating_cleanliness    float64
rating_value          float64
rating_location       float64
price_total             int64
price_original        float64
discount_amount       float64
description            object
has_rating               bool
bedrooms                int64
bathrooms             float64
description_length      int64
has_wifi                 bool
has_pool                 bool
has_pyramid_view         bool
nightly_price         float64
dtype: object

Missing values:
 id                     0
title                  0
url                    0
thumbnail              0
lat                    0
lng                    0
rating_overall         0
reviews_count          0
rating_accuracy      